In [246]:
import random
import numpy as np
import pandas as pd
import math

In [ ]:
dataset = pd.read_csv('../Bases de dados/credit_data.csv')

# Exercicio

O objetivo deste exercício é utilizar outra base de dados para testar as amostragens e comparar os resultados

Faça o download e carregue a base de dados credit_data.csv, que possui informações sobre empréstimos (se o cliente pagará ou não pagará o empréstimo)

Teste cada uma das técnicas de amostragem, selecionando 1000 registros

Para a amostragem estratificada, utilize o atributo c#default para separar as categorias

No final, faça o comparativo da média utilizando os atributos age, income e loan

Na próxima aula você pode assistir a solução para esse exercício

Bom trabalho!

## Técnicas de Amostragem

In [248]:
def amostragem_aleatoria_simples(dataset: pd.DataFrame,
                              qtd_amostras: int) -> pd.DataFrame:
    df = dataset.sample(n=qtd_amostras)

    return df


def amostragem_sistematica(qtd_amostra, df):
    calculo = math.trunc(len(df) // qtd_amostra)
    # random.seed(1)
    valor_inicial = random.randint(0, calculo)
    indices = np.arange(valor_inicial, len(df), step=calculo)

    amostra = df.iloc[indices]
    return amostra



def amostragem_agrupamento(df, qtd_grupos):
    df_copy = df.copy()
    qtd_registros = len(df_copy) // qtd_grupos
    grupos = []
    id_grupo = 1
    contagem = 0
    # Itera sobre todas as linhas do df
    for _ in df_copy.iterrows():
        grupos.append(id_grupo)
        contagem += 1
        if contagem > qtd_registros:
            contagem = 0
            id_grupo += 1

    # Cria a coluna grupos
    df_copy['grupo'] = grupos

    # random.seed(1)
    grupo_sort = random.randint(1,qtd_grupos)
    return df_copy[df_copy['grupo'] == grupo_sort]


# Biblioteca que faz o calculo da amostra estratificada
from sklearn.model_selection import StratifiedShuffleSplit

def amostragem_estratificada(df, porcent, campo):
    # StratifiedShuffleSplit -> Utilizada para buscar a quantidade de registros de uma base. É muito utilizado para treinar modelos de MLops
    split = StratifiedShuffleSplit(test_size=porcent)
    # Primeiro argumento é a base de dados de treinamento e o segundo argumento é a base de dados de teste
    # Se escolhermos test_size = 20%, primeiro argumento terá 80% e o segundo terá 20%
    for _, y in split.split(df, df[campo]):
        df_y = df.iloc[y]

    return df_y


def amostragem_reservatorio(dataset, amostras):
    tamanho = len(dataset)
    stream = []

    for i in range(tamanho):
        stream.append(i)

    i = 0
    reservatorio = [0] * amostras

    for i in range(amostras):
        reservatorio[i] = stream[i]


    while i < tamanho:
        j = random.randrange(i + 1)
        if j < amostras:
            reservatorio[j] = stream[i]
        i += 1

    return dataset.iloc[reservatorio]

## Análises

In [249]:
"""
A base credit_data foi criada com o objetivo de estudar o comportamento financeiro de clientes e prever o risco
de inadimplência em operações de crédito. Ela é comumente utilizada em projetos de aprendizado de máquina e análise preditiva.

A base contém informações como idade, renda, valor do empréstimo e se o cliente ficou inadimplente. Essa amostra contém 2.000 registros.
+++++++++++++++++++++++ METADADOS +++++++++++++++++++++++
    | Coluna     | Descrição                                                                 |
    |------------|---------------------------------------------------------------------------|
    | `clientid` | Identificador único de cada cliente (número sequencial).                  |
    | `income`   | Renda mensal do cliente (em reais ou dólares, dependendo da base usada).  |
    | `age`      | Idade do cliente (em anos).                                               |
    | `loan`     | Valor do empréstimo tomado pelo cliente.                                  |
    | `default`  | Indicador de inadimplência: `1` para inadimplente, `0` para adimplente.   |
"""

dataset.head()

,i#clientid,income,age,loan,c#default
0,1,66155.925095,59.017015,8106.532131,0
1,2,34415.153966,48.117153,6564.745018,0
2,3,57317.170063,63.108049,8020.953296,0
3,4,42709.534201,45.751972,6103.642260,0
4,5,66952.688845,18.584336,8770.099235,1


In [250]:
dataset.shape

(2000, 5)

### Amostragem simples


In [251]:
df_amostra_simpes = amostragem_aleatoria_simples(dataset, 1000)
df_amostra_simpes

,i#clientid,income,age,loan,c#default
1763,1764,45930.452649,49.991331,7765.252827,0
1020,1021,49517.722328,31.549318,7337.950431,1
1311,1312,45311.831839,26.928215,3103.812228,0
1397,1398,45540.325523,59.318140,1490.470251,0
400,401,51625.313230,44.808841,4592.245550,0
...,...,...,...,...,...
1937,1938,25602.957250,28.446377,2214.922493,0
780,781,32720.504799,33.804504,4367.264950,1
839,840,62955.608293,29.549510,207.543818,0
944,945,66255.029528,28.960025,7475.212282,0


### Amostragem sistematica


In [252]:
df_amostra_sistematica = amostragem_sistematica(1000, dataset)
df_amostra_sistematica

,i#clientid,income,age,loan,c#default
0,1,66155.925095,59.017015,8106.532131,0
2,3,57317.170063,63.108049,8020.953296,0
4,5,66952.688845,18.584336,8770.099235,1
6,7,48430.359613,26.809132,5722.581981,0
8,9,40654.892537,55.496853,4755.825280,0
...,...,...,...,...,...
1990,1991,34237.575419,34.101654,2658.090632,0
1992,1993,30803.806165,23.250084,623.024153,0
1994,1995,24254.700791,37.751622,2225.284643,0
1996,1997,69516.127573,23.162104,3503.176156,0


### Amostragem por grupo


In [253]:
df_amostra_grupo = amostragem_agrupamento(dataset, 2)
df_amostra_grupo

,i#clientid,income,age,loan,c#default,grupo
0,1,66155.925095,59.017015,8106.532131,0,1
1,2,34415.153966,48.117153,6564.745018,0,1
2,3,57317.170063,63.108049,8020.953296,0,1
3,4,42709.534201,45.751972,6103.642260,0,1
4,5,66952.688845,18.584336,8770.099235,1,1
...,...,...,...,...,...,...
996,997,49104.768240,35.538517,9452.217947,0,1
997,998,65776.232413,39.798191,2805.863745,0,1
998,999,36192.149452,21.402403,7236.173930,1,1
999,1000,62165.861186,19.602543,4739.948954,0,1


### amostragem estratificada

In [254]:
dataset['c#default'].value_counts()

c#default
0    1717
1     283
Name: count, dtype: int64

In [255]:
# Descobrir qual a quantidade proporcional de registros vamos identificar para cada grupo 0 e 1
qtd_0 = 1717 * 50 / 100
qtd_1 = 283 * 50 / 100
print(f'Quantidade proporcional para o grupo 0 em %: {qtd_0 / len(dataset) * 100}')
print(f'Quantidade proporcional para o grupo 1 em %: {qtd_1 / len(dataset) * 100}')
print(f'Prova real somando os valores: {qtd_0 + qtd_1}')


Quantidade proporcional para o grupo 0 em %: 42.925000000000004
Quantidade proporcional para o grupo 1 em %: 7.074999999999999
Prova real somando os valores: 1000.0


In [256]:
porcentagem = 1000 / len(dataset)
df_amostra_estratificada = amostragem_estratificada(dataset, porcentagem, 'c#default')

In [257]:
df_amostra_estratificada['c#default'].value_counts()

c#default
0    858
1    142
Name: count, dtype: int64

In [258]:
# Podemos observar que a quantidade de amostra foi selecionada de forma distribuida. Os valores em porcentagem batem com os resultados acima
print(f'Quantidade de amostra para o grupo 0 em porcentagem: {(859 / len(dataset)) * 100}%')
print(f'Quantidade de amostra para o grupo 1 em porcentagem {(141 / len(dataset)) * 100}%')

Quantidade de amostra para o grupo 0 em porcentagem: 42.95%
Quantidade de amostra para o grupo 1 em porcentagem 7.049999999999999%


In [259]:
df_amostra_reservatorio = amostragem_reservatorio(dataset, 1000)

### Comparativo

In [260]:
media_compare_age = dataset['age'].mean()
media_compare_income = dataset['income'].mean()
media_compare_loan = dataset['loan'].mean()

In [261]:
# Amostra Simples
media_simples_age = df_amostra_simpes['age'].mean()
media_simples_income = df_amostra_simpes['income'].mean()
media_simples_loan = df_amostra_simpes['loan'].mean()

print(f"Age validacao: {media_compare_age} -> Age simples: {media_simples_age} = diff: {media_compare_age - media_simples_age}")
print(f"Age income: {media_compare_income} -> income simples: {media_simples_income} = diff: {media_compare_income - media_simples_income}")
print(f"Age loan: {media_compare_loan} -> loan simples: {media_simples_loan} = diff: {media_compare_loan - media_simples_loan}")

Age validacao: 40.80755937840458 -> Age simples: 40.58054183867039 = diff: 0.22701753973418448
Age income: 45331.600017793244 -> income simples: 45262.692870316096 = diff: 68.9071474771481
Age loan: 4444.369694688258 -> loan simples: 4478.142077905904 = diff: -33.772383217646166


In [262]:
# Amostra Sistematica
media_sistematica_age = df_amostra_sistematica['age'].mean()
media_sistematica_income = df_amostra_sistematica['income'].mean()
media_sistematica_loan = df_amostra_sistematica['loan'].mean()

print(f"Age validacao: {media_compare_age} -> Age simples: {media_sistematica_age} = diff: {media_compare_age - media_sistematica_age}")
print(f"Age income: {media_compare_income} -> income simples: {media_sistematica_income} = diff: {media_compare_income - media_sistematica_income}")
print(f"Age loan: {media_compare_loan} -> loan simples: {media_sistematica_loan} = diff: {media_compare_loan - media_sistematica_loan}")

Age validacao: 40.80755937840458 -> Age simples: 40.91117381141754 = diff: -0.10361443301296447
Age income: 45331.600017793244 -> income simples: 45691.49875066942 = diff: -359.89873287617957
Age loan: 4444.369694688258 -> loan simples: 4506.78797642633 = diff: -62.41828173807153


In [263]:
# Amostra Estratificada
media_estratificada_age = df_amostra_estratificada['age'].mean()
media_estratificada_income = df_amostra_estratificada['income'].mean()
media_estratificada_loan = df_amostra_estratificada['loan'].mean()

print(f"Age validacao: {media_compare_age} -> Age simples: {media_estratificada_age} = diff: {media_compare_age - media_estratificada_age}")
print(f"Age income: {media_compare_income} -> income simples: {media_estratificada_income} = diff: {media_compare_income - media_estratificada_income}")
print(f"Age loan: {media_compare_loan} -> loan simples: {media_estratificada_loan} = diff: {media_compare_loan - media_estratificada_loan}")

Age validacao: 40.80755937840458 -> Age simples: 41.08109091456188 = diff: -0.2735315361573001
Age income: 45331.600017793244 -> income simples: 45562.329168920405 = diff: -230.72915112716146
Age loan: 4444.369694688258 -> loan simples: 4551.452882053785 = diff: -107.08318736552701


In [264]:
# Amostragem por grupo
media_grupo_age = df_amostra_grupo['age'].mean()
media_grupo_income = df_amostra_grupo['income'].mean()
media_grupo_loan = df_amostra_grupo['loan'].mean()

print(f"Age validacao: {media_compare_age} -> Age simples: {media_grupo_age} = diff: {media_compare_age - media_grupo_age}")
print(f"Age income: {media_compare_income} -> income simples: {media_grupo_income} = diff: {media_compare_income - media_grupo_income}")
print(f"Age loan: {media_compare_loan} -> loan simples: {media_grupo_loan} = diff: {media_compare_loan - media_grupo_loan}")

Age validacao: 40.80755937840458 -> Age simples: 41.0432231120503 = diff: -0.23566373364572257
Age income: 45331.600017793244 -> income simples: 44846.74925986141 = diff: 484.85075793183205
Age loan: 4444.369694688258 -> loan simples: 4390.161493744205 = diff: 54.208200944053715


In [265]:
# Amostragem de reservatório

media_reservatorio_age = df_amostra_reservatorio['age'].mean()
media_reservatorio_income = df_amostra_reservatorio['income'].mean()
media_reservatorio_loan = df_amostra_reservatorio['loan'].mean()

print(f"Age validacao: {media_compare_age} -> Age simples: {media_reservatorio_age} = diff: {media_compare_age - media_reservatorio_age}")
print(f"Age income: {media_compare_income} -> income simples: {media_reservatorio_income} = diff: {media_compare_income - media_reservatorio_income}")
print(f"Age loan: {media_compare_loan} -> loan simples: {media_reservatorio_loan} = diff: {media_compare_loan - media_reservatorio_loan}")

Age validacao: 40.80755937840458 -> Age simples: 41.03968812167441 = diff: -0.23212874326983268
Age income: 45331.600017793244 -> income simples: 45319.888810045086 = diff: 11.711207748157904
Age loan: 4444.369694688258 -> loan simples: 4424.631186123615 = diff: 19.73850856464287
